In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from benchmark import ModelBenchmark
from helpers import DATA_PATH, get_data_from_file

In [4]:
corrupt, clean = get_data_from_file('small')

### Baseline

In [203]:
from symspellpy import symspellpy
import pkg_resources

max_edit_distance = 2
prefix_length = 7

sym_spell = symspellpy.SymSpell(max_edit_distance, prefix_length)
dictionary_path = pkg_resources.resource_filename(
        "symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)
# bigram_path = pkg_resources.resource_filename(
#     "symspellpy", "frequency_bigramdictionary_en_243_342.txt")
# sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

sym_spell.create_dictionary(DATA_PATH + "corpus.txt", encoding='utf-8')
input_phrase = corrupt[0]
suggestions = sym_spell.lookup_compound(input_phrase, max_edit_distance=max_edit_distance)



In [211]:
print(suggestions[0].term)

team number tr 1 p span contents 0


In [15]:
benchmark = ModelBenchmark(device='cpu')

In [ ]:
benchmark.benchmark_model(sym_spell,
                          clean,
                          corrupt,
                          "symspell",
                          lambda model, data: model.correct_string(data),
                          warm_up_runs=0,
                          num_runs=2)


### Neuspell


In [4]:

from neuspell import BertChecker

checker = BertChecker(device='cuda')
checker.from_pretrained()

data folder is set to `c:\users\brumda\documents\neuspell\neuspell\../neuspell_data` script
loading vocab from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise\vocab.pkl
initializing model
loading pretrained weights from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise
Loading model params from checkpoint dir: c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise


In [36]:
benchmark = ModelBenchmark()

#### Bench

In [37]:
benchmark.benchmark_model(checker,
                          corrupt,
                          clean,
                          "neuspell-bert",
                          lambda model, data: model.correct_string(data),
                          warm_up_runs=0,
                          num_runs=2)


Benchmark results:
   Model: neuspell-bert
   Size: 706.53 MB
   Inference Time: 14.68 s
   GPU Memory: 841.85 MB
   RAM Memory: 0.01 MB
   Peak RAM Memory: 14.85 MB
   Throughput: 4.36 sentences/sec
   Throughput: 229.30 ms/sentence
   Throughput: 50.63 tokens/sec
   Accuracy sentences: 3.12%
   Accuracy tokens: 59.62%
   Correct → Correct: 441.0
   Correct → Incorrect: 127.0
   Incorrect → Correct: 2.0
   Incorrect → Incorrect: 173.0
   Recall: 1.14%
   Precision: 1.55%
   F0.5: 1.45%
   Token Correction Rate: 1.14%
   Token Incorrection Rate: 22.36%

In [4]:
corrupt[0]

'team_number = tr[1].p.span.contents[0]'

In [299]:
checker.correct_string(corrupt[5])
# checker.correct_string(" I luk forawd to itd.")

'name: "org.swift.digs.pkg-builder.says-pkg-product-validation",'

In [16]:
clean[4]

'will be those of the redirected or rewritten routes when applicable.'

### FIX

In [287]:
def get_subtokens(tokens, start_index):
    """Get the first word after start index"""
    result = [tokens[start_index]]

    # Find indices of tokens after start_index that start with '##'
    mask = np.char.startswith(tokens[start_index + 1:], '##')

    # Find the first False (non-## token) in the mask
    # Else happens if the rest of the tokens are one word
    non_subtoken_indices = np.where(~mask)[0]
    end_idx = non_subtoken_indices[0] if len(non_subtoken_indices) > 0 else len(mask)

    result.extend(tokens[start_index + 1:start_index + 1 + end_idx])
    return np.array(result)

In [288]:
from transformers import BertTokenizerFast
import numpy as np

index = 4
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
tokenizer.do_basic_tokenize = True
tokenizer.tokenize_chinese_chars = False
test_string = corrupt[index]
test_string_clean = clean[index]
transformed_tokens = checker.correct_string(test_string)
print(f"input: {test_string}", f"output: {transformed_tokens}", sep="\n")

input: will be those of the redirected or rewriten routes when applicable.
output: will be those of the redirected or rewritten routes when applicable .


In [289]:
original_offsets = tokenizer(test_string, return_offsets_mapping=True)['offset_mapping']
tokenization = np.array(tokenizer.tokenize(transformed_tokens))
token_offsets = np.array(original_offsets[1:-1])  # first and last offsets are [CLS] and [SEP]
# Pre-allocate arrays
pretok_sent = np.empty(len(tokenization), dtype=object)
offsets_merged = np.empty((len(tokenization), 2), dtype=int)

idx_offset = 0
out_idx = 0
in_row = 0

care = False
if len(tokenization) != len(original_offsets):
    in_tok = np.array(tokenizer.tokenize(test_string))
    care = True

i = 0
while i < len(tokenization):
    token = tokenization[i]

    if token.startswith("##"):
        in_row += 1
        pretok_sent[out_idx - 1] = pretok_sent[out_idx - 1] + token[2:]
        offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
    else:
        if care and in_row >= 1:
            # check if the previous merged token has correct offsets
            start = i - idx_offset - in_row - 1
            word = get_subtokens(in_tok, start)
            orig_toks = tokenizer.tokenize(''.join(word.replace('##', '') for word in word))

            if len(orig_toks) != in_row + 1:
                # fix the offsets
                for _ in range(len(orig_toks) - (in_row + 1)):
                    offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
                    idx_offset += 1
            in_row = 0

        # handle the new token
        pretok_sent[out_idx] = token
        offsets_merged[out_idx] = token_offsets[i + idx_offset]
        out_idx += 1
    i += 1

# Truncate arrays to actual size
pretok_sent = pretok_sent[:out_idx]
offsets_merged = offsets_merged[:out_idx]

next_starts = np.roll(offsets_merged[:, 0], -1)[:-1]
current_ends = offsets_merged[:-1, 1]
mask = next_starts > current_ends
# boolean mask checking if the offsets align or next start index is greater than previous end
# indicating space in the original text
mask = np.append(mask, False)
tokens_arr = np.array(pretok_sent)
# add spaces back at corresponding places
tokens_with_space = np.where(mask, np.char.add(pretok_sent, ' '), pretok_sent)
reconstructed_text = "".join(tokens_with_space)

In [290]:
print(*original_offsets[1:-1])
print(tokenizer.tokenize(test_string))
print(80 * "#")
print(pretok_sent)
print(offsets_merged)
print(80 * "#")
print(mask)
print(80 * "#")
reconstructed_text

(0, 4) (5, 7) (8, 13) (14, 16) (17, 20) (21, 24) (24, 27) (27, 31) (32, 34) (35, 37) (37, 38) (38, 42) (42, 43) (44, 50) (51, 55) (56, 66) (66, 67)
['will', 'be', 'those', 'of', 'the', 'red', '##ire', '##cted', 'or', 're', '##w', '##rite', '##n', 'routes', 'when', 'applicable', '.']
################################################################################
[np.str_('will') np.str_('be') np.str_('those') np.str_('of')
 np.str_('the') 'redirected' np.str_('or') 'rewritten' np.str_('routes')
 np.str_('when') np.str_('applicable') np.str_('.')]
[[ 0  4]
 [ 5  7]
 [ 8 13]
 [14 16]
 [17 20]
 [21 31]
 [32 34]
 [35 43]
 [44 50]
 [51 55]
 [56 66]
 [66 67]]
################################################################################
[ True  True  True  True  True  True  True  True  True  True False False]
################################################################################


'will be those of the redirected or rewritten routes when applicable.'

### T5

In [11]:
from happytransformer import HappyTextToText

# Load T5 model for grammar/spelling correction
happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")

# Example sentence with typos
input_text = corrupt[0]

# Correct the sentence
output = happy_tt.generate_text(f"grammar: {input_text}")
print(output.text, clean[0], sep='\n')


04/01/2025 15:56:13 - INFO - happytransformer.happy_transformer -   Using device: cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Moving model to cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Initializing a pipeline
Device set to use cuda:0


Team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]


In [1]:
from detect_typo_model import TypoDetectionModel

In [6]:
pred_model = TypoDetectionModel()
pred_model.load_model("pred_typo_models/best_model.pt")

Model loaded from pred_typo_models/best_model.pt


In [16]:
for i in range(20):
    text = corrupt[i]
    print(text, clean[i], sep='\n')
    print(pred_model.predict(text))
    print(80*'-')

team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]
0.47374123334884644
--------------------------------------------------------------------------------
* The internal method that handles the pointer out event from the browser.
* The internal method that handles the pointer over event from the browser.
0.5091391801834106
--------------------------------------------------------------------------------
To understand what is in the `dockercfg` field, convert the secret data to a
To understand what is in the `.dockercfg` field, convert the secret data to a
0.8158062696456909
--------------------------------------------------------------------------------
number of blocks has been removed.  The rpc calls are deprecated and will either
number of blocks has been removed.  The RPC calls are deprecated and will either
0.8373557329177856
--------------------------------------------------------------------------------
will be those of the redirected or rewriten routes w